In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import seaborn as sns

sns.set_theme(style='white')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.titlepad': 14,
    'axes.labelpad': 8,
})

# Faculty abbreviations (used in all plots for readability)
FACULTY_ABBREV = {
    'Faculty of Social Science & Public Policy':                'SSPP',
    'Faculty of Natural, Mathematical & Engineering Sciences':  'NMES',
    'Faculty of Dentistry, Oral & Craniofacial Sciences':       'DOCS',
    'Faculty of Life Sciences & Medicine':                      'LSM',
    'Institute of Psychiatry, Psychology & Neuroscience':       'IoPPN',
    'Faculty of Arts & Humanities':                             'AH',
    'Research Management & Innovation':                         'RMI',
    "King's Business School":                                   'KBS',
    'Students & Education':                                     'SE',
    'The Dickson Poon School of Law':                           'DPSL',
    'IT':                                                       'IT',
    'Other':                                                    'Other',
    'unknown':                                                  'N/A',
}

FACULTY_MERGE = {
    'Faculty of Life Sciences and Medicine':
        'Faculty of Life Sciences & Medicine',
    'IoPPN':
        'Institute of Psychiatry, Psychology & Neuroscience',
    'Dentistry, Oral & Craniofacial Sciences':
        'Faculty of Dentistry, Oral & Craniofacial Sciences',
    'Social Science & Public Policy':
        'Faculty of Social Science & Public Policy',
}

unix_zerotime = pd.to_datetime("1970-01-01 00:00", utc=True)

In [ ]:
## function definitions
def read_job_metrics(input_file):
    schema = {
        'job_id': pl.Int64,
        'username': pl.String,
        'faculty': pl.String,
        'submission_type': pl.String,
        'step_count': pl.Int64,
        'state': pl.Int64,
        'exit_code': pl.Int64,
        'is_success': pl.Int64,
        'time_start': pl.Float64,
        'time_end': pl.Float64,
        'partition': pl.String,
        'elapsed_sec': pl.Float64,
        'wait_sec': pl.Float64,
        'timelimit_sec': pl.Float64,
        'cpu_eff_req': pl.Float64,
        'cpu_eff_alloc': pl.Float64,
        'mem_eff': pl.Float64,
        'mem_eff_alloc': pl.Float64,
        'time_eff': pl.Float64,
        'total_cpu_sec': pl.Float64,
        'user_cpu_sec': pl.Float64,
        'sys_cpu_sec': pl.Float64,
        'user_cpu_pct': pl.Float64,
        'maxrss_bytes': pl.Int128,
        'reqmem_bytes': pl.Int64,
        'allocmem_bytes': pl.Int64,
        'mem_type': pl.String,
        'req_cpus': pl.Int64,
        'alloc_cpus': pl.Int64,
        'req_gpus': pl.Int64,
        'alloc_gpus': pl.Int64,
        'n_nodes': pl.Int64,
        'n_tasks': pl.Int64,
        'submit_line_ntasks': pl.Int64,
        'submit_line_cpus_per_task': pl.Int64,
        'submit_line_interactive': pl.Int64,
    }

    df = pl.read_csv(
        input_file, 
        comment_prefix='#', 
        null_values='NULL',
        schema=schema
        ).with_columns(
            pl.col("faculty").str.strip_chars('"'),
            pl.from_epoch("time_start", time_unit="s").dt.replace_time_zone("UTC"),
            pl.from_epoch("time_end", time_unit="s").dt.replace_time_zone("UTC")
        )
    return df

def fix_faculty_labels(df, faculty_merge, faculty_abbrev):
    df = df.with_columns(
        pl.col("faculty").replace(FACULTY_MERGE),
        pl.col('faculty')
        .replace_strict(FACULTY_ABBREV, default=pl.col('faculty'))
        .alias('faculty_short')
    )
    return df

## Read data since May 2022

In [ ]:
input_files = [
    "../results/data/2022-05-01_2024-08-31_job_level_metrics_all_jobs.csv", 
    "../results/data/2024-08-26_2026-08-26_job_level_metrics_all_jobs.csv", 
    "../results/data/2026-08-25_2026-09-07_job_level_metrics_all_jobs.csv", 
    "../results/data/2026-09-07_2026-10-15_job_level_metrics_all_jobs.csv"
    ]

df_list = [read_job_metrics(input) for input in input_files]
df = pl.concat(df_list).unique()
df = fix_faculty_labels(df, faculty_merge=FACULTY_MERGE, faculty_abbrev=FACULTY_ABBREV)

print(f'Included jobs: {len(df):,}')

In [ ]:
# filter to a 12 month period - last 12 full months in dataset
# keep only jobs with at least 1 GPU requested
df = df.with_columns(
    (pl.col('time_start') - pl.duration(seconds=pl.col('wait_sec'))).alias('time_submit')
    ).filter(
        (pl.col("time_submit") >= pd.to_datetime("2025-10-01", utc=True)) &
        (pl.col("time_submit") < pd.to_datetime("2026-10-01", utc=True))
    ).filter(
        pl.col("req_gpus") > 0
    )

print(f'Included jobs after filtering: {len(df):,}')

In [ ]:
df = df.with_columns(
    gpu_h = pl.col("alloc_gpus") * (pl.col("elapsed_sec") / 360)
)

top10_gpu_users = (df.select("username", "gpu_h")
                   .group_by("username")
                   .sum()
                   .sort("gpu_h", descending=True)
                   .head(10))

print(", ".join(top10_gpu_users["username"]))

In [ ]:
top10_users_top_jobs = df.filter(pl.col("username").is_in(top10_gpu_users["username"].to_list())).select(
        pl.all().top_k_by("gpu_h", k=10).over("username", mapping_strategy="explode")
)
top10_users_top_jobs

In [ ]:
top10_users_top_jobs.write_csv("top10_GPU_users_top_jobs_2025-10_to_2026-09.csv")